# nn.Sequentialと動的なネットワーク構築

---
## 目的
`nn.Sequential`を用いて，複数の層から構成されるネットワークを簡潔に定義する方法を理解する．
また，層の数やユニット数をコンストラクタの引数に応じて動的に変更できるネットワークの実装方法を身につける．

## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
from time import time

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchsummary

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## nn.Sequentialとは

これまでのノートブックでは，`nn.Module`を継承したクラスの`__init__`で層を定義し，`forward`メソッド内でそれらの層を1つずつ順番に呼び出すことでネットワークを実装してきました．

単純に複数の層を上から順番に適用するだけのネットワークであれば，`nn.Sequential`にモジュールを並べて渡すことで，`forward`を自分で書かなくても同じ処理を簡潔に定義できます．
以下では，同じ構造の多層パーセプトロン（MLP）を，`nn.Module`による記述と`nn.Sequential`による記述の両方で実装し，比較します．

In [ ]:
### nn.Moduleでforwardを明示的に記述する場合
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(28*28, 256)
        self.l2 = nn.Linear(256, 128)
        self.l3 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        h = self.relu(self.l1(x))
        h = self.relu(self.l2(h))
        h = self.l3(h)
        return h

### nn.Sequentialで同じ構造を記述する場合
model_sequential = nn.Sequential(
    nn.Linear(28*28, 256),
    nn.ReLU(),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
)

dummy_input = torch.randn(4, 28*28)
print("MLP(nn.Module) output shape:", MLP()(dummy_input).shape)
print("nn.Sequential output shape:", model_sequential(dummy_input).shape)

`nn.Sequential`は`forward`を自分で記述する必要がない分，コード量を減らすことができます．
ただし，途中の層の出力を複数箇所で使い回したり，Skip Connection（ResNetなどで用いられる，層の入力を出力に加算する構造）のように処理が一直線ではないネットワークを実装したりする場合には，`nn.Sequential`だけでは表現できないため，`nn.Module`で`forward`を明示的に記述する必要があります．

## nn.Moduleの中でnn.Sequentialを利用する

実際には，`nn.Sequential`単体で使うのではなく，`nn.Module`を継承したクラスの中で，複数の層をまとめる部分にのみ`nn.Sequential`を利用するという書き方がよく使われます．
このようにすることで，`nn.Sequential`による簡潔さを保ちながら，必要に応じて`forward`側で追加の処理（`view`による形状変更など）を記述できます．

以下では，これまでのノートブックで使用してきたCNNを，`self.features`に畳み込み部分を，`self.classifier`に全結合部分をそれぞれ`nn.Sequential`としてまとめる形に書き直します．

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 4, kernel_size=3, stride=1, padding=1),
            nn.Sigmoid(),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(14*14*4, 10),
        )

    def forward(self, x):
        h = self.features(x)
        h = h.view(h.size(0), -1)
        h = self.classifier(h)
        return h

model = CNN()
model = model.to(device)
torchsummary.summary(model, (1, 28, 28), device=device.type)

## 動的なネットワーク構築

ここまでの実装では，層の数やユニット数（チャンネル数）はすべてクラスの中に直接書き込まれていました（ハードコーディング）．
しかし，層の数やユニット数を変えた複数のネットワークを試したいことがよくあります．そのたびにクラスを定義し直すのは非効率です．

そこで，層の数やユニット数をコンストラクタの引数（リストなど）として受け取り，`for`文で層を1つずつ`list`に追加してから，最後に`nn.Sequential(*layers)`としてまとめる，という実装方法がよく使われます．
以下では，中間層のユニット数のリスト`hidden_units`を引数として受け取り，任意の層数のMLPを構築できる`DynamicMLP`を実装します．

In [ ]:
class DynamicMLP(nn.Module):
    def __init__(self, input_dim, hidden_units, num_classes):
        super().__init__()

        layers = []
        in_dim = input_dim
        for units in hidden_units:
            layers.append(nn.Linear(in_dim, units))
            layers.append(nn.ReLU())
            in_dim = units
        layers.append(nn.Linear(in_dim, num_classes))

        self.layers = nn.Sequential(*layers)

    def forward(self, x):
        return self.layers(x)

### 中間層の構成を変えて，複数のネットワークを作成
for hidden_units in [[128], [256, 128], [512, 256, 128, 64]]:
    model = DynamicMLP(input_dim=28*28, hidden_units=hidden_units, num_classes=10)
    num_params = sum(p.numel() for p in model.parameters())
    print("hidden_units: {}, パラメータ数: {}".format(hidden_units, num_params))
    print(model)
    print()

## 動的なCNNの構築

同じ考え方を畳み込みニューラルネットワークにも適用できます．各畳み込み層の出力チャンネル数のリスト`channel_list`を引数として受け取り，任意の層数のCNNを構築する`DynamicCNN`を実装します．

畳み込み層の数によって，最後の特徴マップの縦横のサイズ（`height`, `width`）が変化してしまうため，全結合層への入力サイズを層数ごとに計算し直す必要があります．
これを避けるため，最後に`nn.AdaptiveAvgPool2d(1)`を挿入します．これは，入力される特徴マップの縦横のサイズによらず，チャンネルごとに平均を取ることで，常に`1×1`のサイズへ変換してくれる層です．これにより，畳み込み層の数を変えても全結合層の実装を変更せずに済みます．

In [ ]:
class DynamicCNN(nn.Module):
    def __init__(self, in_channels, channel_list, num_classes):
        super().__init__()

        layers = []
        c_in = in_channels
        for c_out in channel_list:
            layers.append(nn.Conv2d(c_in, c_out, kernel_size=3, stride=1, padding=1))
            layers.append(nn.ReLU())
            layers.append(nn.MaxPool2d(2, 2))
            c_in = c_out
        layers.append(nn.AdaptiveAvgPool2d(1))

        self.features = nn.Sequential(*layers)
        self.classifier = nn.Linear(channel_list[-1], num_classes)

    def forward(self, x):
        h = self.features(x)
        h = h.view(h.size(0), -1)
        h = self.classifier(h)
        return h

### チャンネル数の構成を変えて，複数のネットワークを作成
for channel_list in [[8], [8, 16], [8, 16, 32]]:
    model = DynamicCNN(in_channels=1, channel_list=channel_list, num_classes=10)
    model = model.to(device)
    num_params = sum(p.numel() for p in model.parameters())
    print("channel_list: {}, パラメータ数: {}".format(channel_list, num_params))

### 構造の異なるネットワークの学習・比較

`DynamicCNN`を用いて構造の異なる複数のネットワークを作成し，MNISTデータセットを用いて実際に学習・評価し，構造の違いによる精度やパラメータ数の違いを比較します．

In [ ]:
train_data = torchvision.datasets.MNIST(root="./", train=True, transform=transforms.ToTensor(), download=True)
test_data = torchvision.datasets.MNIST(root="./", train=False, transform=transforms.ToTensor(), download=True)

train_loader = torch.utils.data.DataLoader(train_data, batch_size=100, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=100, shuffle=False)

criterion = nn.CrossEntropyLoss()
criterion = criterion.to(device)

for channel_list in [[8], [8, 16], [8, 16, 32]]:
    model = DynamicCNN(in_channels=1, channel_list=channel_list, num_classes=10)
    model = model.to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)

    model.train()
    train_start = time()
    for epoch in range(3):
        for image, label in train_loader:
            image = image.to(device)
            label = label.to(device)

            y = model(image)
            loss = criterion(y, label)

            model.zero_grad()
            loss.backward()
            optimizer.step()

    model.eval()
    count = 0
    with torch.no_grad():
        for image, label in test_loader:
            image = image.to(device)
            label = label.to(device)
            y = model(image)
            pred = torch.argmax(y, dim=1)
            count += torch.sum(pred == label)
    test_acc = count.item() / len(test_data)

    num_params = sum(p.numel() for p in model.parameters())
    print("channel_list: {}, パラメータ数: {}, test accuracy: {:.4f}, elapsed time: {:.2f} sec".format(
        channel_list, num_params, test_acc, time() - train_start))

## 課題

1. `DynamicMLP`または`DynamicCNN`を改良し，活性化関数の種類（`nn.ReLU`, `nn.Sigmoid`など）も引数で指定できるようにしてみましょう．

2. `DynamicMLP`または`DynamicCNN`の各層の間に`nn.Dropout`を挿入できるように改良してみましょう．

    ヒント：`dropout_rate`を引数として受け取り，`0`より大きい場合のみ`layers.append(nn.Dropout(dropout_rate))`のように層のリストに追加します．